# Chapter 2 — Sweep and optimize a coupled grounded LC

Engineer course · source candidate · CONVERGING

# Chapter 2 — Sweep and optimize a coupled grounded LC

This complete Chapter is its own clean-kernel execution unit. Its four
web Lessons are shorter reading views over these same ordered source
fragments; they are not independent notebooks. QMD is the editable
authority, while `chapter.ipynb` is a generated zero-output transport
artifact.

## Lesson 1 — Build a parameterized coupled LC

### Begin from a clean Chapter kernel

Follow the [engineer course setup](../setup.qmd), then open this
Chapter’s `chapter.ipynb`, restart its Python kernel, and use **Run
All**. Chapter 2 rebuilds the circuit below; it does not continue
Chapter 1’s kernel.

Both C and L are independent physical inputs because the optional final
Lesson varies both. The coupling capacitor and terminated Port remain
fixed literals.

In [ ]:
from IPython.display import display

from scnsim import (
    CMAESSpec,
    CircuitDiagramSpec,
    CircuitPlan,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    OptimizationSpec,
    OptimizationVariable,
    ParameterDefinitions,
    ParameterSet,
    ParameterSpace,
    ParameterSpec,
    ReductionPipeline,
    components,
    units as u,
)

inputs = ParameterDefinitions(id="engineer_lc_sweep_design")
capacitance = inputs.parameter(
    id="capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
inductance = inputs.parameter(
    id="inductance",
    baseline=5.8 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)

A definition becomes part of this Plan only when a real component field
consumes it. The Subsystem owns the reusable, readable LC unit; the
coupler and Port remain parent-owned connections to its environment.

In [ ]:
plan = CircuitPlan(id="engineer_swept_coupled_lc")
resonator = plan.subsystem(id="resonator")

capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=inductance)
)
resonator_bus = resonator.bus(id="signal")
resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)
resonator_terminal = resonator.expose_pin(
    id="terminal",
    at=resonator_bus,
)
resonator_coordinate = resonator.expose_coordinate(
    id="signal",
    at=resonator_bus,
)

The Pin is the parent’s wiring boundary. The Coordinate selects that
same child signal Bus for analysis; it is not a wire endpoint and adds
no electrical node.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
coupling_capacitor = plan.add(
    components.capacitor(id="coupling_capacitor", capacitance=6.0 * u.fF)
)
plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_capacitor,),
    end=resonator_terminal,
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

The 6 fF coupling branch ends directly at the public child Pin. No
analysis-only root Bus or zero-component Link is required.

## Lesson 2 — Sweep capacitance

**Prerequisite:** run Lesson 1 in the aggregate Chapter Notebook.

Create one sealed Run and retain the child’s published Coordinate. The 6
GHz root hint selects the deterministic root basin; it is not a sweep
bound or the answer.

In [ ]:
run = CircuitRun(
    plan=plan,
    workspace="workspaces/engineer-chapter-02",
)
root_view = run.original.reduce(
    ReductionPipeline().retain(resonator_coordinate)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_coordinate,
    root_hint=6.0 * u.GHz,
)

This one-axis grid preserves the declared 100, 110, and 120 fF order. L
is an explicit fixed input at 5.8 nH, not a hidden second axis.

In [ ]:
capacitance_space = ParameterSpace.grid(
    axes={
        capacitance: (
            100.0 * u.fF,
            110.0 * u.fF,
            120.0 * u.fF,
        ),
    },
    fixed=ParameterSet({inductance: 5.8 * u.nH}),
)

In [ ]:
capacitance_sweep = run.evaluate(
    root_view,
    root_spec,
    parameters=capacitance_space,
)

Every point retains its complete effective `ParameterSet`, source index,
identity, and either a typed root Result or typed failure. Collection
reads a quantity already requested by `root_spec`; it launches no new
calculation.

In [ ]:
display(capacitance_sweep.show())
capacitance_frequency = capacitance_sweep.collect(
    quantity=root_spec.frequency
)
capacitance_sweep_figure = capacitance_frequency.show(x=capacitance)
capacitance_sweep_figure

This is the Chapter’s core stopping point: one physical Plan, one
fixed-L capacitance space, and one typed Result per declared point. The
next Lesson is an optional tuning workflow, not a condition for
understanding the sweep.

## Lesson 3 — Optimize and verify one winner

**Prerequisite:** run Lessons 1–2 in the aggregate Chapter Notebook.

> **This request may take several minutes**
>
> The declared CMA-ES request evaluates real candidates with a fixed
> seed and budget. Let it finish; the website never launches it.

The optimization varies only C from 80 to 140 fF. L remains fixed at 5.8
nH, the loaded-frequency target is 6.2 GHz, and the existing
deterministic CMA-ES controls remain `seed=17` and
`max_evaluations=200`.

In [ ]:
optimization_spec = OptimizationSpec(
    variables=(
        OptimizationVariable(
            parameter=capacitance,
            bounds=(80.0 * u.fF, 140.0 * u.fF),
        ),
    ),
    objectives=(
        CostObjective(
            id="resonance_frequency",
            quantity=root_spec.frequency,
            target=6.2 * u.GHz,
            weight=1.0 * u.dimensionless,
        ),
    ),
    optimizer=CMAESSpec(seed=17, max_evaluations=200),
)
fixed_inductance = ParameterSet({inductance: 5.8 * u.nH})

In [ ]:
optimization_spec.show()

The specification fixes the active input, bounds, objective, seed, and
budget before execution. It does not claim that any candidate meets a
design target.

In [ ]:
optimization = run.optimize(
    root_view,
    optimization_spec,
    parameters=fixed_inductance,
)
best_parameters = optimization.best.parameters
display(best_parameters)
display(optimization.best.cost)

Use the returned winner in a separate root evaluation. This request
neither mutates the Plan’s baselines nor infers a value from the
optimizer’s cost.

In [ ]:
winner_root = run.evaluate(
    root_view,
    root_spec,
    parameters=best_parameters,
)
display(winner_root.frequency)
display(winner_root.linewidth)
winner_root.show()

Render that exact returned `ParameterSet` on the same Plan. The
schematic is authoring evidence at the selected point, not numerical
proof of the objective.

In [ ]:
winner_diagram = plan.render_schematic(
    CircuitDiagramSpec(),
    parameters=best_parameters,
)

In [ ]:
winner_diagram.show()
winner_diagram.audit.show()

## Lesson 4 — Optional C/L spaces

**Prerequisite:** run Lessons 1–3 in the aggregate Chapter Notebook.

This optional extension reuses the same Plan, Run, View, and root
request. A 3 × 2 Cartesian grid evaluates every declared C/L
combination. The listed space evaluates only the two explicit pairs and
does not become a grid.

In [ ]:
capacitance_inductance_grid = ParameterSpace.grid(
    axes={
        capacitance: (
            100.0 * u.fF,
            110.0 * u.fF,
            120.0 * u.fF,
        ),
        inductance: (
            5.8 * u.nH,
            6.0 * u.nH,
        ),
    },
    fixed=ParameterSet(),
)
listed_pairs = ParameterSpace.points(
    (
        ParameterSet({
            capacitance: 100.0 * u.fF,
            inductance: 5.8 * u.nH,
        }),
        ParameterSet({
            capacitance: 120.0 * u.fF,
            inductance: 6.0 * u.nH,
        }),
    )
)

In [ ]:
optional_grid_sweep = run.evaluate(
    root_view,
    root_spec,
    parameters=capacitance_inductance_grid,
)

In [ ]:
display(optional_grid_sweep.show())
optional_grid_frequency = optional_grid_sweep.collect(
    quantity=root_spec.frequency
)
optional_grid_figure = optional_grid_frequency.show(
    x=capacitance,
    y=inductance,
)
optional_grid_figure

In [ ]:
listed_pair_sweep = run.evaluate(
    root_view,
    root_spec,
    parameters=listed_pairs,
)
listed_pair_sweep.show()

The Cartesian and listed spaces retain different order and identity.
Failed points, if any, remain typed failures in their exact positions;
they are never silently replaced, interpolated, or reported as zero.